# IEEE 33-Bus Graph Transformer Dataset Pipeline (HDF5)

This notebook builds a reproducible, scenario-based power-flow dataset for the IEEE 33-bus distribution system. Each sample combines load perturbations and a feasible radial switch configuration, solves power flow, extracts graph features, and streams the result to HDF5 for Graph Transformer research.

**Research targets**
- Active/reactive load variation impact prediction
- Switching behavior modeling
- Contingency and reconfiguration analysis
- Graph Transformer labels: losses, voltage profile, and line loading

**Storage target**: `data/ieee33_dataset.h5`

In [ ]:
from pathlib import Path
import copy
import json

import pandapower as pp
import pandapower.networks as pn
import networkx as nx
import numpy as np
import pandas as pd
from tqdm import tqdm
from joblib import Parallel, delayed
import h5py
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt

GLOBAL_SEED = 20260622
N_SAMPLES = 1000
BATCH_SIZE = 64
N_JOBS = -1
H5_PATH = Path("data/ieee33_dataset.h5")
COMPRESSION = "gzip"
COMPRESSION_OPTS = 4
VOLTAGE_LIMITS_PU = (0.95, 1.05)
OVERLOAD_LIMIT_PERCENT = 100.0

np.random.seed(GLOBAL_SEED)
plt.rcParams["figure.figsize"] = (9, 5)

print(f"Configuration ready | samples={N_SAMPLES:,} | batch_size={BATCH_SIZE} | seed={GLOBAL_SEED}")

## 1. IEEE-33 Network Loading and Switchable Tie-Line Preparation

`pandapower.networks.case33bw()` is the authoritative base network. The helper below preserves the base radial feeder and ensures the five standard IEEE-33 normally-open tie branches are represented as switchable out-of-service lines. This makes branch-exchange sampling deterministic and explicit.

In [ ]:
# Standard IEEE-33 normally-open tie branches, zero-based bus numbering.
# Values are the conventional Baran-Wu tie branch impedances used when the
# pandapower case representation does not already include these branches.
STANDARD_TIE_BRANCH_DATA = [
    {"from_bus": 20, "to_bus": 7, "r_ohm": 2.0, "x_ohm": 2.0},
    {"from_bus": 8, "to_bus": 14, "r_ohm": 2.0, "x_ohm": 2.0},
    {"from_bus": 11, "to_bus": 21, "r_ohm": 2.0, "x_ohm": 2.0},
    {"from_bus": 17, "to_bus": 32, "r_ohm": 0.5, "x_ohm": 0.5},
    {"from_bus": 24, "to_bus": 28, "r_ohm": 0.5, "x_ohm": 0.5},
]


def _ensure_line_columns(net):
    """Ensure line status and tie markers exist with stable boolean semantics."""
    if "in_service" not in net.line.columns:
        net.line["in_service"] = True
    net.line["in_service"] = net.line["in_service"].fillna(True).astype(bool)

    if "is_tie" not in net.line.columns:
        net.line["is_tie"] = False
    net.line["is_tie"] = net.line["is_tie"].fillna(False).astype(bool)


def _line_mask_for_endpoints(net, from_bus, to_bus):
    from_values = net.line["from_bus"].astype(int).to_numpy()
    to_values = net.line["to_bus"].astype(int).to_numpy()
    return ((from_values == int(from_bus)) & (to_values == int(to_bus))) | ((from_values == int(to_bus)) & (to_values == int(from_bus)))


def _median_or_default(series, default_value):
    if series is None or len(series) == 0:
        return float(default_value)
    value = float(pd.Series(series).dropna().median())
    return value if np.isfinite(value) else float(default_value)


def _ensure_ieee33_tie_lines(net):
    """Add or mark the five standard IEEE-33 tie branches as normally open."""
    _ensure_line_columns(net)
    max_i_ka = _median_or_default(net.line["max_i_ka"] if "max_i_ka" in net.line.columns else None, 0.4)
    c_nf_per_km = _median_or_default(net.line["c_nf_per_km"] if "c_nf_per_km" in net.line.columns else None, 0.0)

    for tie in STANDARD_TIE_BRANCH_DATA:
        mask = _line_mask_for_endpoints(net, tie["from_bus"], tie["to_bus"])
        matching_indices = list(net.line.index[mask])

        if matching_indices:
            net.line.loc[matching_indices, "is_tie"] = True
            net.line.loc[matching_indices, "in_service"] = False
            continue

        line_idx = pp.create_line_from_parameters(
            net,
            from_bus=tie["from_bus"],
            to_bus=tie["to_bus"],
            length_km=1.0,
            r_ohm_per_km=tie["r_ohm"],
            x_ohm_per_km=tie["x_ohm"],
            c_nf_per_km=c_nf_per_km,
            max_i_ka=max_i_ka,
            name=f"tie_{tie['from_bus'] + 1}_{tie['to_bus'] + 1}",
            in_service=False,
        )
        net.line.at[line_idx, "is_tie"] = True

    net.line["in_service"] = net.line["in_service"].astype(bool)
    net.line["is_tie"] = net.line["is_tie"].astype(bool)
    return net


def get_root_bus(net):
    """Return the substation/root bus from ext_grid when available."""
    if len(net.ext_grid) > 0 and "bus" in net.ext_grid.columns:
        return int(net.ext_grid.iloc[0]["bus"])
    return int(net.bus.index.min())


def load_ieee33():
    """Load IEEE-33, prepare switchable ties, and return pandapower net plus base graph."""
    net = pn.case33bw()
    _ensure_ieee33_tie_lines(net)
    base_graph = build_graph(net, active_only=True)
    if not validate_configuration(net):
        raise ValueError("Prepared IEEE-33 base configuration is not radial and connected.")
    return net, base_graph

## 2. Graph Builder and Topology Features

The graph builder supports two modes:
- `active_only=True` for physical feasibility checks on the currently energized radial feeder.
- `active_only=False` for feature extraction over all physical branches, where `switch_status` identifies open tie or sectionalizing branches.

In [ ]:
def _bus_load_table(net):
    """Aggregate active load at each bus in MW/Mvar."""
    loads = pd.DataFrame(index=net.bus.index, data={"p_load": 0.0, "q_load": 0.0})
    if len(net.load) == 0:
        return loads

    load_df = net.load.copy()
    if "in_service" in load_df.columns:
        load_df = load_df[load_df["in_service"].fillna(True).astype(bool)]
    if len(load_df) == 0:
        return loads

    grouped = load_df.groupby("bus")[["p_mw", "q_mvar"]].sum()
    loads.loc[grouped.index, "p_load"] = grouped["p_mw"].astype(float)
    loads.loc[grouped.index, "q_load"] = grouped["q_mvar"].astype(float)
    return loads


def _initial_voltage_pu_table(net):
    """Flat-start voltage profile, using ext_grid setpoints at substation buses."""
    voltage = pd.Series(1.0, index=net.bus.index, dtype=float)
    if len(net.ext_grid) > 0:
        for _, row in net.ext_grid.iterrows():
            vm_pu = float(row["vm_pu"]) if "vm_pu" in net.ext_grid.columns else 1.0
            voltage.loc[int(row["bus"])] = vm_pu
    return voltage


def line_is_closed(net, line_idx):
    """Return whether a line is electrically closed, combining line and switch status."""
    closed = bool(net.line.at[line_idx, "in_service"])
    if "switch" in net and len(net.switch) > 0:
        switches = net.switch[(net.switch["et"] == "l") & (net.switch["element"].astype(int) == int(line_idx))]
        if len(switches) > 0:
            closed = closed and bool(switches["closed"].fillna(True).astype(bool).all())
    return bool(closed)


def _line_impedance(net, line_idx):
    row = net.line.loc[line_idx]
    length = float(row["length_km"]) if "length_km" in net.line.columns else 1.0
    r = float(row["r_ohm_per_km"]) * length if "r_ohm_per_km" in net.line.columns else 0.0
    x = float(row["x_ohm_per_km"]) * length if "x_ohm_per_km" in net.line.columns else 0.0
    return r, x


def build_graph(net, active_only=False):
    """Convert a pandapower network to a NetworkX graph with node and edge features."""
    _ensure_line_columns(net)
    G = nx.Graph()
    load_table = _bus_load_table(net)
    voltage_table = _initial_voltage_pu_table(net)
    root_bus = get_root_bus(net)

    for bus_idx in net.bus.index:
        bus_idx_int = int(bus_idx)
        G.add_node(
            bus_idx_int,
            p_load=float(load_table.loc[bus_idx, "p_load"]),
            q_load=float(load_table.loc[bus_idx, "q_load"]),
            bus_type=1.0 if bus_idx_int == root_bus else 0.0,
            voltage=float(voltage_table.loc[bus_idx]),
        )

    for line_idx, row in net.line.iterrows():
        switch_status = 1.0 if line_is_closed(net, line_idx) else 0.0
        if active_only and switch_status < 0.5:
            continue
        r, x = _line_impedance(net, line_idx)
        G.add_edge(
            int(row["from_bus"]),
            int(row["to_bus"]),
            line_idx=int(line_idx),
            r=float(r),
            x=float(x),
            impedance=float(np.sqrt(r * r + x * x)),
            switch_status=float(switch_status),
            is_tie=bool(row["is_tie"]),
        )
    return G


def compute_feeder_depth(G, root_bus):
    """Compute BFS feeder depth from the substation/root bus."""
    depth = {int(node): -1 for node in G.nodes}
    if root_bus not in G.nodes:
        return depth
    for node, value in nx.single_source_shortest_path_length(G, int(root_bus)).items():
        depth[int(node)] = int(value)
    return depth


def compute_distance_to_substation(G, root_bus):
    """Compute impedance-weighted shortest-path distance to the substation/root bus."""
    distance = {int(node): np.inf for node in G.nodes}
    if root_bus not in G.nodes:
        return distance
    try:
        lengths = nx.single_source_dijkstra_path_length(G, int(root_bus), weight="impedance")
    except Exception:
        lengths = nx.single_source_shortest_path_length(G, int(root_bus))
    for node, value in lengths.items():
        distance[int(node)] = float(value)
    return distance

## 3. Load Perturbations, Radiality Checks, and Branch-Exchange Switching

Each sample independently scales bus loads and then performs a single feasible branch exchange: close one normally-open tie line and open one branch on the induced cycle. Invalid configurations are discarded before any power-flow solve.

In [ ]:
def check_connectivity(G):
    """Return True when all buses are connected in the active graph."""
    return len(G.nodes) > 0 and nx.is_connected(G)


def check_radiality(G):
    """Return True when the active graph is radial. For connected graphs this is nx.is_tree."""
    return len(G.nodes) > 0 and nx.is_tree(G)


def validate_configuration(net):
    """Validate the energized feeder topology independently of the power-flow solver."""
    active_graph = build_graph(net, active_only=True)
    return check_connectivity(active_graph) and check_radiality(active_graph)


def perturb_loads(net, min_scale=0.8, max_scale=1.2, rng=None):
    """Scale each bus load independently and return per-bus scaling factors."""
    if rng is None:
        rng = np.random.default_rng(GLOBAL_SEED)

    scales = pd.Series(1.0, index=net.bus.index, dtype=float)
    if len(net.load) == 0:
        return scales.to_numpy(dtype=np.float32)

    load_buses = sorted(pd.unique(net.load["bus"].astype(int)))
    for bus_idx in load_buses:
        scales.loc[bus_idx] = float(rng.uniform(min_scale, max_scale))

    for load_idx, row in net.load.iterrows():
        scale = float(scales.loc[int(row["bus"])])
        net.load.at[load_idx, "p_mw"] = float(row["p_mw"]) * scale
        net.load.at[load_idx, "q_mvar"] = float(row["q_mvar"]) * scale

    return scales.to_numpy(dtype=np.float32)


def _switch_configuration_key(net):
    """Stable key from active line indices for uniqueness analysis."""
    active_lines = [str(int(idx)) for idx in net.line.index if line_is_closed(net, idx)]
    return ",".join(active_lines)


def _cycle_edges_from_nodes(net, cycle_nodes):
    cycle_set = set(int(node) for node in cycle_nodes)
    cycle_edges = []
    for line_idx, row in net.line.iterrows():
        if not line_is_closed(net, line_idx):
            continue
        if int(row["from_bus"]) in cycle_set and int(row["to_bus"]) in cycle_set:
            cycle_edges.append(int(line_idx))
    return cycle_edges


def generate_switch_configuration(net, rng=None, max_attempts=100):
    """Generate a feasible branch-exchange configuration.

    The sampler closes one normally-open tie line, finds the resulting cycle,
    opens one non-tie branch on that cycle, and validates that the final active
    topology is connected and radial.
    """
    if rng is None:
        rng = np.random.default_rng(GLOBAL_SEED)

    _ensure_ieee33_tie_lines(net)
    original_status = net.line["in_service"].astype(bool).copy()
    tie_lines = [int(idx) for idx in net.line.index[net.line["is_tie"].astype(bool)]]
    inactive_ties = [idx for idx in tie_lines if not line_is_closed(net, idx)]

    if len(inactive_ties) == 0:
        raise ValueError("No inactive tie lines are available for branch exchange.")

    for attempt in range(max_attempts):
        net.line["in_service"] = original_status.copy()

        closed_tie = int(rng.choice(inactive_ties))
        net.line.at[closed_tie, "in_service"] = True

        trial_graph = build_graph(net, active_only=True)
        cycles = nx.cycle_basis(trial_graph)
        if len(cycles) == 0:
            continue

        closed_tie_row = net.line.loc[closed_tie]
        tie_endpoints = {int(closed_tie_row["from_bus"]), int(closed_tie_row["to_bus"])}
        selected_cycle = None
        for cycle in cycles:
            if tie_endpoints.issubset(set(int(node) for node in cycle)):
                selected_cycle = cycle
                break
        if selected_cycle is None:
            selected_cycle = cycles[0]

        cycle_edges = _cycle_edges_from_nodes(net, selected_cycle)
        open_candidates = [idx for idx in cycle_edges if idx != closed_tie and not bool(net.line.at[idx, "is_tie"])]
        if len(open_candidates) == 0:
            continue

        opened_branch = int(rng.choice(open_candidates))
        net.line.at[opened_branch, "in_service"] = False

        if validate_configuration(net):
            opened_row = net.line.loc[opened_branch]
            summary = {
                "closed_tie_line": int(closed_tie),
                "closed_tie_edge": [int(closed_tie_row["from_bus"]), int(closed_tie_row["to_bus"])],
                "opened_branch_line": int(opened_branch),
                "opened_branch_edge": [int(opened_row["from_bus"]), int(opened_row["to_bus"])],
                "attempts": int(attempt + 1),
                "config_key": _switch_configuration_key(net),
            }
            return summary

    net.line["in_service"] = original_status.copy()
    raise ValueError(f"Failed to generate a feasible branch exchange after {max_attempts} attempts.")

## 4. Power Flow Engine and Graph Feature Extraction

Power-flow failures are returned as structured failures. Feature extraction uses the active topology only for depth/distance calculations and stores all physical line features with switch status.

In [ ]:
NODE_FEATURE_NAMES = ["p_load_mw", "q_load_mvar", "depth", "distance_to_substation"]
EDGE_FEATURE_NAMES = ["r_ohm", "x_ohm", "switch_status"]
GLOBAL_FEATURE_NAMES = ["total_load_mw", "num_buses", "num_lines"]
LABEL_NAMES = ["total_loss_kw", "min_voltage_pu", "max_loading_percent", "violations"]


def run_powerflow(net):
    """Run pandapower power flow and return robust scalar labels."""
    try:
        pp.runpp(
            net,
            algorithm="nr",
            init="flat",
            calculate_voltage_angles=False,
            enforce_q_lims=False,
            max_iteration=50,
            tolerance_mva=1e-8,
            numba=False,
        )
    except Exception as exc:
        return {"success": False, "reason": f"powerflow_failed: {type(exc).__name__}: {exc}"}

    if not bool(getattr(net, "converged", False)):
        return {"success": False, "reason": "powerflow_not_converged"}

    total_loss_kw = 0.0
    if "res_line" in net and len(net.res_line) > 0 and "pl_mw" in net.res_line.columns:
        total_loss_kw += float(np.nan_to_num(net.res_line["pl_mw"].to_numpy(dtype=float), nan=0.0).sum() * 1000.0)
    if "res_trafo" in net and len(net.res_trafo) > 0 and "pl_mw" in net.res_trafo.columns:
        total_loss_kw += float(np.nan_to_num(net.res_trafo["pl_mw"].to_numpy(dtype=float), nan=0.0).sum() * 1000.0)

    vm_pu = np.nan_to_num(net.res_bus["vm_pu"].to_numpy(dtype=float), nan=np.nan)
    loading = np.array([0.0], dtype=float)
    if "res_line" in net and len(net.res_line) > 0 and "loading_percent" in net.res_line.columns:
        loading = np.nan_to_num(net.res_line["loading_percent"].to_numpy(dtype=float), nan=0.0, posinf=0.0, neginf=0.0)

    v_min, v_max = VOLTAGE_LIMITS_PU
    voltage_violation_count = int(((vm_pu < v_min) | (vm_pu > v_max)).sum())
    overload_count = int((loading > OVERLOAD_LIMIT_PERCENT).sum())

    return {
        "success": True,
        "total_loss_kw": float(total_loss_kw),
        "min_voltage_pu": float(np.nanmin(vm_pu)),
        "max_line_loading_percent": float(np.nanmax(loading)),
        "voltage_violation_count": voltage_violation_count,
        "overload_count": overload_count,
        "violations": int(voltage_violation_count + overload_count),
    }


def extract_graph_features(net, G=None):
    """Extract node, edge, and global feature matrices for one solved scenario."""
    if G is None:
        G = build_graph(net, active_only=False)
    active_graph = build_graph(net, active_only=True)
    root_bus = get_root_bus(net)
    depth = compute_feeder_depth(active_graph, root_bus)
    distance = compute_distance_to_substation(active_graph, root_bus)
    load_table = _bus_load_table(net)

    node_features = []
    for bus_idx in net.bus.index:
        bus_idx_int = int(bus_idx)
        node_features.append([
            float(load_table.loc[bus_idx, "p_load"]),
            float(load_table.loc[bus_idx, "q_load"]),
            float(depth.get(bus_idx_int, -1)),
            float(distance.get(bus_idx_int, np.inf)),
        ])

    edge_features = []
    for line_idx, _ in net.line.iterrows():
        r, x = _line_impedance(net, line_idx)
        edge_features.append([
            float(r),
            float(x),
            1.0 if line_is_closed(net, line_idx) else 0.0,
        ])

    total_load = float(load_table["p_load"].sum())
    global_features = np.array([
        total_load,
        float(len(net.bus)),
        float(len(net.line)),
    ], dtype=np.float32)

    return {
        "node_features": np.asarray(node_features, dtype=np.float32),
        "edge_features": np.asarray(edge_features, dtype=np.float32),
        "global_features": global_features,
    }

## 5. Single-Sample Generator and Streaming HDF5 Storage

Workers generate compact per-sample arrays and metadata. HDF5 writes are serialized in the notebook process to avoid concurrent file access issues.

In [ ]:
BASE_NET = None
BASE_GRAPH = None


def initialize_base_network():
    """Initialize global base objects once for reproducible sample generation."""
    global BASE_NET, BASE_GRAPH
    BASE_NET, BASE_GRAPH = load_ieee33()
    print(
        "Base IEEE-33 ready | "
        f"buses={len(BASE_NET.bus)} | lines={len(BASE_NET.line)} | "
        f"active_lines={int(BASE_NET.line['in_service'].sum())} | ties={int(BASE_NET.line['is_tie'].sum())}"
    )
    return BASE_NET, BASE_GRAPH


def _copy_base_network():
    """Deep-copy the base network without mutating BASE_NET."""
    if BASE_NET is None:
        initialize_base_network()
    return copy.deepcopy(BASE_NET)


def generate_sample(seed):
    """Generate one scenario: perturb loads, reconfigure switches, validate, solve, extract."""
    rng = np.random.default_rng(int(seed))
    try:
        net = _copy_base_network()
        load_scaling = perturb_loads(net, min_scale=0.8, max_scale=1.2, rng=rng)
        switch_summary = generate_switch_configuration(net, rng=rng)

        if not validate_configuration(net):
            return {"success": False, "seed": int(seed), "reason": "invalid_configuration"}

        pf_result = run_powerflow(net)
        if not pf_result["success"]:
            return {"success": False, "seed": int(seed), "reason": pf_result["reason"]}

        graph = build_graph(net, active_only=False)
        features = extract_graph_features(net, graph)
        labels = np.array([
            pf_result["total_loss_kw"],
            pf_result["min_voltage_pu"],
            pf_result["max_line_loading_percent"],
            pf_result["violations"],
        ], dtype=np.float32)

        metadata = {
            "seed": int(seed),
            "load_scaling_factors": [float(value) for value in load_scaling],
            "switch_action_summary": switch_summary,
            "switch_config_key": switch_summary["config_key"],
            "voltage_violation_count": int(pf_result["voltage_violation_count"]),
            "overload_count": int(pf_result["overload_count"]),
        }

        return {
            "success": True,
            "seed": int(seed),
            "node_features": features["node_features"],
            "edge_features": features["edge_features"],
            "global_features": features["global_features"],
            "labels": labels,
            "metadata": metadata,
        }
    except Exception as exc:
        return {"success": False, "seed": int(seed), "reason": f"sample_failed: {type(exc).__name__}: {exc}"}


def initialize_hdf5(path, overwrite=True):
    """Create an HDF5 file and write dataset-level metadata."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    mode = "w" if overwrite else "a"
    with h5py.File(path, mode, track_order=True) as h5:
        h5.attrs["schema_version"] = "1.0"
        h5.attrs["dataset_name"] = "ieee33_graph_transformer_dataset"
        h5.attrs["global_seed"] = int(GLOBAL_SEED)
        h5.attrs["node_feature_names"] = json.dumps(NODE_FEATURE_NAMES)
        h5.attrs["edge_feature_names"] = json.dumps(EDGE_FEATURE_NAMES)
        h5.attrs["global_feature_names"] = json.dumps(GLOBAL_FEATURE_NAMES)
        h5.attrs["label_names"] = json.dumps(LABEL_NAMES)
        h5.attrs["total_attempted"] = 0
        h5.attrs["valid_samples"] = 0
        h5.attrs["rejected_samples"] = 0
        h5.attrs["rejection_reasons"] = json.dumps({})
    return path


def _write_array(group, name, value):
    group.create_dataset(
        name,
        data=np.asarray(value),
        compression=COMPRESSION,
        compression_opts=COMPRESSION_OPTS,
        shuffle=True,
    )


def write_records_to_hdf5(path, records, start_sample_id):
    """Append valid records to HDF5 with one group per sample."""
    string_dtype = h5py.string_dtype(encoding="utf-8")
    next_id = int(start_sample_id)
    with h5py.File(path, "a", track_order=True) as h5:
        for record in records:
            group_name = f"sample_{next_id:06d}"
            group = h5.create_group(group_name)
            _write_array(group, "node_features", record["node_features"].astype(np.float32))
            _write_array(group, "edge_features", record["edge_features"].astype(np.float32))
            _write_array(group, "global_features", record["global_features"].astype(np.float32))
            _write_array(group, "labels", record["labels"].astype(np.float32))
            group.create_dataset("metadata", data=json.dumps(record["metadata"]), dtype=string_dtype)
            group.attrs["seed"] = int(record["seed"])
            group.attrs["switch_config_key"] = record["metadata"]["switch_config_key"]
            group.attrs["total_load_mw"] = float(record["global_features"][0])
            next_id += 1
    return next_id


def update_hdf5_report(path, total_attempted, valid_samples, rejected_samples, rejection_reasons):
    with h5py.File(path, "a") as h5:
        h5.attrs["total_attempted"] = int(total_attempted)
        h5.attrs["valid_samples"] = int(valid_samples)
        h5.attrs["rejected_samples"] = int(rejected_samples)
        h5.attrs["rejection_reasons"] = json.dumps(rejection_reasons)

## 6. Parallel Dataset Generation

Default generation creates `N_SAMPLES = 1000` scenarios. To scale later, increase `N_SAMPLES` (for example to `100_000`) and tune `BATCH_SIZE`/`N_JOBS` for the host CPU. The pipeline writes valid samples immediately after each batch and never holds the full graph dataset in RAM.

## 6A. Research-Grade Diversity, Stress, Scoring, and HDF5 Upgrade

This section upgrades the baseline generator into a controllable dataset system. It adds balanced load regimes, multi-mode topology exploration, topology deduplication, stress-case targeting, validity scoring, online diversity tracking, and the publication-oriented HDF5 schema requested for Graph Transformer training.

In [ ]:
LOAD_REGIMES = {
    "normal": {"range": (0.9, 1.1), "weight": 0.50},
    "heavy": {"range": (1.1, 1.4), "weight": 0.30},
    "light": {"range": (0.5, 0.9), "weight": 0.20},
}
SWITCH_MODES = ["local_perturbation", "multi_step_random_walk", "constrained_exploration"]
SWITCH_MODE_WEIGHTS = np.array([0.35, 0.40, 0.25], dtype=float)
SWITCH_MODE_WEIGHTS = SWITCH_MODE_WEIGHTS / SWITCH_MODE_WEIGHTS.sum()
MIN_SWITCH_DIVERSITY = 0.30
TARGET_SWITCH_DIVERSITY = 0.50
MAX_CONFIG_REPEAT = 3
MAX_TOPOLOGY_REPEAT = 3
MIN_VALIDITY_SCORE = 0.35
MIN_STRESS_VALIDITY_SCORE = 0.18
STRESS_TARGET_FRACTION = 0.20
STRESS_VOLTAGE_THRESHOLD = 0.85
STRESS_LOADING_THRESHOLD = 90.0
MAX_ATTEMPT_MULTIPLIER = 25

NODE_FEATURE_NAMES = [
    "p_load_mw",
    "q_load_mvar",
    "feeder_depth",
    "electrical_distance_to_substation",
    "normalized_load",
    "bus_type",
    "initial_voltage_pu",
]
EDGE_FEATURE_NAMES = [
    "r_ohm",
    "x_ohm",
    "switch_status",
    "impedance_magnitude",
    "power_flow_estimate_mw",
]
GLOBAL_FEATURE_NAMES = [
    "total_load_mw",
    "num_buses",
    "num_lines",
    "active_lines",
    "stress_case_flag",
]
LABEL_NAMES = ["loss", "min_voltage", "max_loading"]


def _normalize_line_current_limits(net, nominal_max_i_ka=0.40):
    """Replace placeholder ampacity values with realistic limits for loading labels."""
    if "max_i_ka" not in net.line.columns:
        net.line["max_i_ka"] = nominal_max_i_ka
    max_i = net.line["max_i_ka"].astype(float)
    invalid = (~np.isfinite(max_i)) | (max_i <= 0.0) | (max_i > 5.0)
    net.line.loc[invalid, "max_i_ka"] = float(nominal_max_i_ka)
    return net


def initialize_base_network():
    """Initialize base IEEE-33 with realistic loading limits for publication-quality labels."""
    global BASE_NET, BASE_GRAPH
    BASE_NET, BASE_GRAPH = load_ieee33()
    _normalize_line_current_limits(BASE_NET, nominal_max_i_ka=0.40)
    BASE_GRAPH = build_graph(BASE_NET, active_only=True)
    print(
        "Base IEEE-33 ready | "
        f"buses={len(BASE_NET.bus)} | lines={len(BASE_NET.line)} | "
        f"active_lines={int(BASE_NET.line['in_service'].sum())} | ties={int(BASE_NET.line['is_tie'].sum())} | "
        f"median_max_i_ka={float(BASE_NET.line['max_i_ka'].median()):.3f}"
    )
    return BASE_NET, BASE_GRAPH


def sample_load_mode(rng):
    modes = list(LOAD_REGIMES.keys())
    weights = np.array([LOAD_REGIMES[mode]["weight"] for mode in modes], dtype=float)
    weights = weights / weights.sum()
    return str(rng.choice(modes, p=weights))


def perturb_loads(net, load_mode="normal", rng=None, correlated_probability=0.35):
    """Balanced per-bus load perturbation with optional correlated structure."""
    if rng is None:
        rng = np.random.default_rng(GLOBAL_SEED)
    if load_mode not in LOAD_REGIMES:
        raise ValueError(f"Unknown load_mode={load_mode!r}; expected one of {sorted(LOAD_REGIMES)}")

    low, high = LOAD_REGIMES[load_mode]["range"]
    scales = pd.Series(1.0, index=net.bus.index, dtype=float)
    if len(net.load) == 0:
        return scales.to_numpy(dtype=np.float32), {"correlated": False, "correlation_strength": 0.0}

    load_buses = sorted(pd.unique(net.load["bus"].astype(int)))
    independent = pd.Series(rng.uniform(low, high, size=len(load_buses)), index=load_buses, dtype=float)
    correlated = bool(rng.uniform() < correlated_probability)
    correlation_strength = 0.0

    if correlated:
        active_graph = build_graph(net, active_only=True)
        depth = compute_feeder_depth(active_graph, get_root_bus(net))
        max_depth = max([value for value in depth.values() if value >= 0] + [1])
        feeder_bias = pd.Series(
            [depth.get(int(bus), 0) / max(max_depth, 1) for bus in load_buses],
            index=load_buses,
            dtype=float,
        )
        global_shift = float(rng.uniform(low, high))
        correlation_strength = float(rng.uniform(0.15, 0.35))
        depth_component = low + (high - low) * feeder_bias
        independent = (
            (1.0 - correlation_strength) * independent
            + 0.5 * correlation_strength * global_shift
            + 0.5 * correlation_strength * depth_component
        ).clip(low, high)

    for bus_idx in load_buses:
        scales.loc[int(bus_idx)] = float(independent.loc[int(bus_idx)])

    for load_idx, row in net.load.iterrows():
        scale = float(scales.loc[int(row["bus"])])
        net.load.at[load_idx, "p_mw"] = float(row["p_mw"]) * scale
        net.load.at[load_idx, "q_mvar"] = float(row["q_mvar"]) * scale

    return scales.to_numpy(dtype=np.float32), {
        "correlated": correlated,
        "correlation_strength": float(correlation_strength),
    }


def generate_stress_case(net, rng=None):
    """Create explicit stress scenarios for undervoltage, high loading, and boundary cases."""
    if rng is None:
        rng = np.random.default_rng(GLOBAL_SEED)

    stress_mode = str(rng.choice(["undervoltage", "overload", "boundary"], p=[0.45, 0.35, 0.20]))
    scales = pd.Series(1.0, index=net.bus.index, dtype=float)
    active_graph = build_graph(net, active_only=True)
    depth = compute_feeder_depth(active_graph, get_root_bus(net))
    max_depth = max([value for value in depth.values() if value >= 0] + [1])

    if stress_mode == "undervoltage":
        base_low, base_high = 1.45, 2.20
        downstream_gain = 0.35
        line_derate = 1.0
    elif stress_mode == "overload":
        base_low, base_high = 1.25, 1.85
        downstream_gain = 0.15
        line_derate = float(rng.uniform(0.55, 0.85))
    else:
        base_low, base_high = 1.10, 1.55
        downstream_gain = 0.20
        line_derate = float(rng.uniform(0.80, 1.0))

    if len(net.load) > 0:
        load_buses = sorted(pd.unique(net.load["bus"].astype(int)))
        for bus_idx in load_buses:
            depth_factor = depth.get(int(bus_idx), 0) / max(max_depth, 1)
            scale = float(rng.uniform(base_low, base_high) * (1.0 + downstream_gain * depth_factor))
            scales.loc[int(bus_idx)] = scale
        for load_idx, row in net.load.iterrows():
            scale = float(scales.loc[int(row["bus"])])
            net.load.at[load_idx, "p_mw"] = float(row["p_mw"]) * scale
            net.load.at[load_idx, "q_mvar"] = float(row["q_mvar"]) * scale

    if "max_i_ka" in net.line.columns and line_derate < 1.0:
        net.line["max_i_ka"] = net.line["max_i_ka"].astype(float) * line_derate

    return scales.to_numpy(dtype=np.float32), {
        "stress_mode": stress_mode,
        "line_rating_multiplier": float(line_derate),
        "correlated": True,
        "correlation_strength": float(downstream_gain),
    }


def graph_signature(net):
    """Topology signature from sorted energized edge endpoints."""
    edge_list = []
    for line_idx, row in net.line.iterrows():
        if line_is_closed(net, line_idx):
            edge = tuple(sorted((int(row["from_bus"]), int(row["to_bus"]))))
            edge_list.append(edge)
    return str(abs(hash(tuple(sorted(edge_list)))))


def load_signature(load_scaling_vector, decimals=2):
    rounded = tuple(np.round(np.asarray(load_scaling_vector, dtype=float), decimals=decimals).tolist())
    return str(abs(hash(rounded)))


def _active_switch_key(net):
    return ",".join(str(int(idx)) for idx in net.line.index if line_is_closed(net, idx))


def _apply_branch_exchange(net, rng):
    open_lines = [int(idx) for idx in net.line.index if not line_is_closed(net, idx)]
    if len(open_lines) == 0:
        raise ValueError("No open branches are available for branch exchange.")

    closed_line = int(rng.choice(open_lines))
    net.line.at[closed_line, "in_service"] = True
    trial_graph = build_graph(net, active_only=True)
    cycles = nx.cycle_basis(trial_graph)
    if len(cycles) == 0:
        net.line.at[closed_line, "in_service"] = False
        raise ValueError("Closing candidate branch did not create a cycle.")

    closed_row = net.line.loc[closed_line]
    closed_endpoints = {int(closed_row["from_bus"]), int(closed_row["to_bus"])}
    selected_cycle = None
    for cycle in cycles:
        if closed_endpoints.issubset(set(int(node) for node in cycle)):
            selected_cycle = cycle
            break
    if selected_cycle is None:
        selected_cycle = cycles[0]

    cycle_edges = _cycle_edges_from_nodes(net, selected_cycle)
    open_candidates = [idx for idx in cycle_edges if idx != closed_line]
    if len(open_candidates) == 0:
        net.line.at[closed_line, "in_service"] = False
        raise ValueError("No branch can be opened after closing candidate branch.")

    opened_line = int(rng.choice(open_candidates))
    net.line.at[opened_line, "in_service"] = False
    if not validate_configuration(net):
        net.line.at[opened_line, "in_service"] = True
        net.line.at[closed_line, "in_service"] = False
        raise ValueError("Branch exchange produced an invalid topology.")

    opened_row = net.line.loc[opened_line]
    return {
        "closed_line": int(closed_line),
        "closed_edge": [int(closed_row["from_bus"]), int(closed_row["to_bus"])],
        "opened_line": int(opened_line),
        "opened_edge": [int(opened_row["from_bus"]), int(opened_row["to_bus"])],
    }


def generate_diverse_switch_config(net, history=None, mode="local_perturbation", rng=None, max_attempts=150, repeat_threshold=MAX_CONFIG_REPEAT):
    """Generate a diverse radial topology using one of three exploration modes."""
    if rng is None:
        rng = np.random.default_rng(GLOBAL_SEED)
    if history is None:
        history = {"config_counts": {}, "topology_counts": {}}
    if mode not in SWITCH_MODES:
        raise ValueError(f"Unknown switch mode={mode!r}; expected one of {SWITCH_MODES}")

    original_status = net.line["in_service"].astype(bool).copy()
    for attempt in range(max_attempts):
        net.line["in_service"] = original_status.copy()
        actions = []
        try:
            if mode == "local_perturbation":
                steps = 1
            elif mode == "multi_step_random_walk":
                steps = int(rng.integers(2, 4))
            else:
                steps = int(rng.choice([1, 2, 3], p=[0.25, 0.45, 0.30]))

            for _ in range(steps):
                actions.append(_apply_branch_exchange(net, rng))

            config_key = _active_switch_key(net)
            topology_id = graph_signature(net)
            config_count = int(history.get("config_counts", {}).get(config_key, 0))
            topology_count = int(history.get("topology_counts", {}).get(topology_id, 0))

            if mode == "constrained_exploration" and (
                config_count >= repeat_threshold or topology_count >= repeat_threshold
            ):
                continue

            return {
                "mode": mode,
                "actions": actions,
                "steps": int(steps),
                "attempts": int(attempt + 1),
                "switch_config_key": config_key,
                "config_hash": topology_id,
                "topology_id": topology_id,
                "prior_config_count": config_count,
                "prior_topology_count": topology_count,
            }
        except Exception:
            continue

    net.line["in_service"] = original_status.copy()
    raise ValueError(f"Failed to generate diverse switch configuration in mode={mode}.")


def _power_flow_estimates(net, active_graph, root_bus):
    load_table = _bus_load_table(net)
    estimates = {}
    for line_idx, row in net.line.iterrows():
        if not line_is_closed(net, line_idx):
            estimates[int(line_idx)] = 0.0
            continue
        u, v = int(row["from_bus"]), int(row["to_bus"])
        graph_copy = active_graph.copy()
        if graph_copy.has_edge(u, v):
            graph_copy.remove_edge(u, v)
        components = list(nx.connected_components(graph_copy))
        downstream = None
        for component in components:
            if root_bus not in component:
                downstream = component
                break
        if downstream is None:
            downstream = set()
        estimates[int(line_idx)] = float(load_table.loc[list(downstream), "p_load"].sum()) if downstream else 0.0
    return estimates


def extract_graph_features(net, G=None):
    """Research-grade graph features with topology and normalized load context."""
    if G is None:
        G = build_graph(net, active_only=False)
    active_graph = build_graph(net, active_only=True)
    root_bus = get_root_bus(net)
    depth = compute_feeder_depth(active_graph, root_bus)
    distance = compute_distance_to_substation(active_graph, root_bus)
    load_table = _bus_load_table(net)
    max_load = max(float(load_table["p_load"].max()), 1e-9)
    flow_estimates = _power_flow_estimates(net, active_graph, root_bus)

    node_features = []
    for bus_idx in net.bus.index:
        bus_idx_int = int(bus_idx)
        p_load = float(load_table.loc[bus_idx, "p_load"])
        q_load = float(load_table.loc[bus_idx, "q_load"])
        node_features.append([
            p_load,
            q_load,
            float(depth.get(bus_idx_int, -1)),
            float(distance.get(bus_idx_int, np.inf)),
            float(p_load / max_load),
            1.0 if bus_idx_int == root_bus else 0.0,
            float(_initial_voltage_pu_table(net).loc[bus_idx]),
        ])

    edge_features = []
    for line_idx, _ in net.line.iterrows():
        r, x = _line_impedance(net, line_idx)
        edge_features.append([
            float(r),
            float(x),
            1.0 if line_is_closed(net, line_idx) else 0.0,
            float(np.sqrt(r * r + x * x)),
            float(flow_estimates.get(int(line_idx), 0.0)),
        ])

    total_load = float(load_table["p_load"].sum())
    global_features = np.array([
        total_load,
        float(len(net.bus)),
        float(len(net.line)),
        float(sum(1 for idx in net.line.index if line_is_closed(net, idx))),
        0.0,
    ], dtype=np.float32)

    return {
        "node_features": np.asarray(node_features, dtype=np.float32),
        "edge_features": np.asarray(edge_features, dtype=np.float32),
        "global_features": global_features,
    }


def compute_validity_score(pf_result, topology_id, history, stress_case=False):
    min_voltage = float(pf_result["min_voltage_pu"])
    max_loading = float(pf_result["max_line_loading_percent"])
    voltage_quality = float(np.clip((min_voltage - 0.75) / (0.98 - 0.75), 0.0, 1.0))
    if min_voltage > 1.05:
        voltage_quality *= float(np.clip((1.10 - min_voltage) / 0.05, 0.0, 1.0))
    loading_quality = float(np.clip((160.0 - max_loading) / 160.0, 0.0, 1.0))
    prior_count = int(history.get("topology_counts", {}).get(topology_id, 0))
    diversity_bonus = float(1.0 / (1.0 + prior_count))
    score = 0.4 * voltage_quality + 0.3 * loading_quality + 0.3 * diversity_bonus
    if stress_case:
        score = max(score, 0.18 + 0.2 * diversity_bonus)
    return {
        "score": float(score),
        "voltage_quality": voltage_quality,
        "loading_quality": loading_quality,
        "diversity_bonus": diversity_bonus,
    }


def _hard_soft_constraints_pass(pf_result, stress_case=False):
    if not pf_result.get("success", False):
        return False, pf_result.get("reason", "powerflow_failed")
    min_voltage = float(pf_result["min_voltage_pu"])
    max_loading = float(pf_result["max_line_loading_percent"])
    if min_voltage < 0.60 or min_voltage > 1.12:
        return False, "extreme_voltage_out_of_bounds"
    if max_loading > 250.0:
        return False, "extreme_overload_out_of_bounds"
    if not stress_case and (min_voltage < 0.82 or max_loading > 180.0):
        return False, "nonstress_soft_limit_exceeded"
    return True, "accepted"


def generate_sample(seed, switch_mode=None, load_mode=None, stress_case=False, history_snapshot=None):
    """Generate one candidate sample with balanced loads, diverse switching, scoring, and signatures."""
    rng = np.random.default_rng(int(seed))
    history_snapshot = history_snapshot or {"config_counts": {}, "topology_counts": {}}
    try:
        net = _copy_base_network()
        if switch_mode is None:
            switch_mode = str(rng.choice(SWITCH_MODES, p=SWITCH_MODE_WEIGHTS))
        if stress_case:
            load_mode = "stress"
            load_scaling, load_details = generate_stress_case(net, rng=rng)
        else:
            load_mode = load_mode or sample_load_mode(rng)
            load_scaling, load_details = perturb_loads(net, load_mode=load_mode, rng=rng)

        switch_summary = generate_diverse_switch_config(net, history_snapshot, mode=switch_mode, rng=rng)
        if not validate_configuration(net):
            return {"success": False, "seed": int(seed), "reason": "invalid_configuration"}

        pf_result = run_powerflow(net)
        hard_pass, hard_reason = _hard_soft_constraints_pass(pf_result, stress_case=stress_case)
        if not hard_pass:
            return {"success": False, "seed": int(seed), "reason": hard_reason}

        topology_id = graph_signature(net)
        config_key = _active_switch_key(net)
        config_hash = switch_summary["config_hash"]
        scoring = compute_validity_score(pf_result, topology_id, history_snapshot, stress_case=stress_case)
        min_score = MIN_STRESS_VALIDITY_SCORE if stress_case else MIN_VALIDITY_SCORE
        if scoring["score"] < min_score:
            return {"success": False, "seed": int(seed), "reason": "low_validity_score"}

        if stress_case:
            is_stressful = (
                float(pf_result["min_voltage_pu"]) < STRESS_VOLTAGE_THRESHOLD
                or float(pf_result["max_line_loading_percent"]) > STRESS_LOADING_THRESHOLD
            )
            if not is_stressful:
                return {"success": False, "seed": int(seed), "reason": "stress_target_not_met"}

        features = extract_graph_features(net, build_graph(net, active_only=False))
        features["global_features"][-1] = 1.0 if stress_case else 0.0
        labels = {
            "loss": float(pf_result["total_loss_kw"]),
            "min_voltage": float(pf_result["min_voltage_pu"]),
            "max_loading": float(pf_result["max_line_loading_percent"]),
        }
        load_sig = load_signature(load_scaling)
        metadata = {
            "seed": int(seed),
            "validity_score": float(scoring["score"]),
            "score_components": scoring,
            "switch_mode": switch_mode,
            "switch_action_summary": switch_summary,
            "switch_config_key": config_key,
            "config_hash": config_hash,
            "topology_id": topology_id,
            "load_signature": load_sig,
            "load_mode": load_mode,
            "load_details": load_details,
            "stress_case": bool(stress_case),
            "voltage_violation_count": int(pf_result["voltage_violation_count"]),
            "overload_count": int(pf_result["overload_count"]),
        }
        return {
            "success": True,
            "seed": int(seed),
            "node_features": features["node_features"],
            "edge_features": features["edge_features"],
            "global_features": features["global_features"],
            "labels": labels,
            "load_scaling": load_scaling.astype(np.float32),
            "metadata": metadata,
        }
    except Exception as exc:
        return {"success": False, "seed": int(seed), "reason": f"sample_failed: {type(exc).__name__}: {exc}"}


def initialize_hdf5(path, overwrite=True):
    """Create the publication-oriented nested HDF5 schema."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    mode = "w" if overwrite else "a"
    with h5py.File(path, mode, track_order=True) as h5:
        h5.attrs["schema_version"] = "2.0"
        h5.attrs["dataset_name"] = "ieee33_diverse_graph_transformer_dataset"
        h5.attrs["global_seed"] = int(GLOBAL_SEED)
        h5.attrs["node_feature_names"] = json.dumps(NODE_FEATURE_NAMES)
        h5.attrs["edge_feature_names"] = json.dumps(EDGE_FEATURE_NAMES)
        h5.attrs["global_feature_names"] = json.dumps(GLOBAL_FEATURE_NAMES)
        h5.attrs["label_names"] = json.dumps(LABEL_NAMES)
        h5.attrs["load_regimes"] = json.dumps(LOAD_REGIMES)
        h5.attrs["switch_modes"] = json.dumps(SWITCH_MODES)
        h5.attrs["stress_target_fraction"] = float(STRESS_TARGET_FRACTION)
        h5.attrs["min_switch_diversity"] = float(MIN_SWITCH_DIVERSITY)
        h5.attrs["target_switch_diversity"] = float(TARGET_SWITCH_DIVERSITY)
        h5.attrs["total_attempted"] = 0
        h5.attrs["valid_samples"] = 0
        h5.attrs["rejected_samples"] = 0
        h5.attrs["rejection_reasons"] = json.dumps({})
        h5.attrs["quality_report"] = json.dumps({})
    return path


def _write_string(group, name, value):
    group.create_dataset(name, data=str(value), dtype=h5py.string_dtype(encoding="utf-8"))


def write_records_to_hdf5(path, records, start_sample_id):
    """Stream accepted records using the nested /graph /topology /scenario /labels /metadata layout."""
    next_id = int(start_sample_id)
    with h5py.File(path, "a", track_order=True) as h5:
        for record in records:
            sample_name = f"sample_{next_id:06d}"
            sample = h5.create_group(sample_name)
            graph_group = sample.create_group("graph")
            topology_group = sample.create_group("topology")
            scenario_group = sample.create_group("scenario")
            labels_group = sample.create_group("labels")
            metadata_group = sample.create_group("metadata")
            metadata = record["metadata"]

            _write_array(graph_group, "node_features", record["node_features"].astype(np.float32))
            _write_array(graph_group, "edge_features", record["edge_features"].astype(np.float32))
            _write_array(graph_group, "global_features", record["global_features"].astype(np.float32))

            _write_string(topology_group, "config_hash", metadata["config_hash"])
            _write_string(topology_group, "topology_id", metadata["topology_id"])
            _write_string(topology_group, "switch_config_key", metadata["switch_config_key"])

            _write_string(scenario_group, "load_mode", metadata["load_mode"])
            _write_array(scenario_group, "load_scaling_vector", record["load_scaling"].astype(np.float32))
            _write_string(scenario_group, "load_signature", metadata["load_signature"])
            scenario_group.create_dataset("stress_case", data=np.asarray(metadata["stress_case"], dtype=np.bool_))

            labels_group.create_dataset("loss", data=np.float32(record["labels"]["loss"]))
            labels_group.create_dataset("min_voltage", data=np.float32(record["labels"]["min_voltage"]))
            labels_group.create_dataset("max_loading", data=np.float32(record["labels"]["max_loading"]))

            metadata_group.create_dataset("seed", data=np.int64(record["seed"]))
            metadata_group.create_dataset("validity_score", data=np.float32(metadata["validity_score"]))
            _write_string(metadata_group, "switch_mode", metadata["switch_mode"])
            _write_string(metadata_group, "json", json.dumps(metadata))

            sample.attrs["seed"] = int(record["seed"])
            sample.attrs["topology_id"] = metadata["topology_id"]
            sample.attrs["config_hash"] = metadata["config_hash"]
            sample.attrs["switch_config_key"] = metadata["switch_config_key"]
            sample.attrs["load_mode"] = metadata["load_mode"]
            sample.attrs["stress_case"] = bool(metadata["stress_case"])
            sample.attrs["validity_score"] = float(metadata["validity_score"])
            sample.attrs["loss"] = float(record["labels"]["loss"])
            sample.attrs["min_voltage"] = float(record["labels"]["min_voltage"])
            sample.attrs["max_loading"] = float(record["labels"]["max_loading"])
            sample.attrs["total_load_mw"] = float(record["global_features"][0])
            sample.attrs["violations"] = int(metadata["voltage_violation_count"] + metadata["overload_count"])
            next_id += 1
    return next_id


def _entropy_from_counts(counts):
    values = np.asarray(list(counts.values()), dtype=float)
    if values.size == 0 or values.sum() <= 0:
        return 0.0
    probabilities = values / values.sum()
    return float(-(probabilities * np.log2(probabilities + 1e-12)).sum())


def _histogram_entropy(values, bins=20):
    values = np.asarray(values, dtype=float)
    if values.size == 0:
        return 0.0
    counts, _ = np.histogram(values, bins=bins)
    counts = counts[counts > 0]
    if counts.size == 0:
        return 0.0
    probabilities = counts / counts.sum()
    return float(-(probabilities * np.log2(probabilities + 1e-12)).sum())


def _quality_report_from_state(state):
    valid = int(state.get("valid_samples", 0))
    attempted = int(state.get("total_attempted", 0))
    rejected = int(state.get("rejected_samples", 0))
    config_counts = state.get("config_counts", {})
    topology_counts = state.get("topology_counts", {})
    switch_diversity = len(config_counts) / max(valid, 1)
    stress_pct = 100.0 * int(state.get("stress_samples", 0)) / max(valid, 1)
    return {
        "total_samples_generated": int(attempted),
        "valid_samples": valid,
        "rejected_samples": rejected,
        "rejection_rate_percent": float(100.0 * rejected / max(attempted, 1)),
        "convergence_rate_percent": float(100.0 * state.get("converged_candidates", 0) / max(attempted, 1)),
        "switch_diversity_score": float(switch_diversity),
        "topology_entropy": _entropy_from_counts(topology_counts),
        "voltage_distribution_entropy": _histogram_entropy(state.get("min_voltage_values", [])),
        "loading_distribution_entropy": _histogram_entropy(state.get("max_loading_values", [])),
        "voltage_distribution_stats": pd.Series(state.get("min_voltage_values", []), dtype=float).describe().to_dict() if state.get("min_voltage_values") else {},
        "loading_distribution_stats": pd.Series(state.get("max_loading_values", []), dtype=float).describe().to_dict() if state.get("max_loading_values") else {},
        "stress_case_percentage": float(stress_pct),
        "unique_switch_configurations": int(len(config_counts)),
        "unique_topologies": int(len(topology_counts)),
        "minimum_switch_diversity_met": bool(switch_diversity >= MIN_SWITCH_DIVERSITY),
        "target_switch_diversity_met": bool(switch_diversity >= TARGET_SWITCH_DIVERSITY),
    }


def update_hdf5_report(path, state):
    report = _quality_report_from_state(state)
    with h5py.File(path, "a") as h5:
        h5.attrs["total_attempted"] = int(state.get("total_attempted", 0))
        h5.attrs["valid_samples"] = int(state.get("valid_samples", 0))
        h5.attrs["rejected_samples"] = int(state.get("rejected_samples", 0))
        h5.attrs["rejection_reasons"] = json.dumps(state.get("rejection_reasons", {}))
        h5.attrs["quality_report"] = json.dumps(report)
    return report


def _history_snapshot(state):
    return {
        "config_counts": dict(state.get("config_counts", {})),
        "topology_counts": dict(state.get("topology_counts", {})),
    }


def _candidate_plan(state, rng):
    valid = int(state.get("valid_samples", 0))
    target_stress_count = int(np.ceil(STRESS_TARGET_FRACTION * int(state.get("target_samples", N_SAMPLES))))
    stress_case = int(state.get("stress_samples", 0)) < target_stress_count
    if not stress_case:
        stress_case = bool(rng.uniform() < STRESS_TARGET_FRACTION)
    switch_mode = str(rng.choice(SWITCH_MODES, p=SWITCH_MODE_WEIGHTS))
    if valid > 0:
        current_diversity = len(state.get("config_counts", {})) / max(valid, 1)
        if current_diversity < TARGET_SWITCH_DIVERSITY:
            switch_mode = "constrained_exploration"
    load_mode = "stress" if stress_case else sample_load_mode(rng)
    return switch_mode, load_mode, stress_case


def _accept_record(record, state):
    metadata = record["metadata"]
    config_key = metadata["switch_config_key"]
    topology_id = metadata["topology_id"]
    config_count = int(state["config_counts"].get(config_key, 0))
    topology_count = int(state["topology_counts"].get(topology_id, 0))
    if config_count >= MAX_CONFIG_REPEAT:
        return False, "config_repeat_limit"
    if topology_count >= MAX_TOPOLOGY_REPEAT:
        return False, "topology_repeat_limit"
    return True, "accepted"


def _record_acceptance(record, state):
    metadata = record["metadata"]
    config_key = metadata["switch_config_key"]
    topology_id = metadata["topology_id"]
    state["config_counts"][config_key] = int(state["config_counts"].get(config_key, 0)) + 1
    state["topology_counts"][topology_id] = int(state["topology_counts"].get(topology_id, 0)) + 1
    state["load_signature_counts"][metadata["load_signature"]] = int(state["load_signature_counts"].get(metadata["load_signature"], 0)) + 1
    state["min_voltage_values"].append(float(record["labels"]["min_voltage"]))
    state["max_loading_values"].append(float(record["labels"]["max_loading"]))
    state["loss_values"].append(float(record["labels"]["loss"]))
    state["total_load_values"].append(float(record["global_features"][0]))
    if metadata["stress_case"]:
        state["stress_samples"] += 1
    if record["labels"]["min_voltage"] < STRESS_VOLTAGE_THRESHOLD or record["labels"]["max_loading"] > STRESS_LOADING_THRESHOLD:
        state["boundary_samples"] += 1
    state["valid_samples"] += 1


def generate_dataset(n_samples=N_SAMPLES, h5_path=H5_PATH, batch_size=BATCH_SIZE, n_jobs=N_JOBS, overwrite=True):
    """Generate exactly n_samples accepted records using safe parallel candidate batches and a single HDF5 writer."""
    initialize_base_network()
    h5_path = initialize_hdf5(h5_path, overwrite=overwrite)
    rng = np.random.default_rng(GLOBAL_SEED)
    max_attempts = int(n_samples * MAX_ATTEMPT_MULTIPLIER)
    next_seed = int(GLOBAL_SEED)
    next_sample_id = 1
    state = {
        "target_samples": int(n_samples),
        "total_attempted": 0,
        "valid_samples": 0,
        "rejected_samples": 0,
        "converged_candidates": 0,
        "stress_samples": 0,
        "boundary_samples": 0,
        "rejection_reasons": {},
        "config_counts": {},
        "topology_counts": {},
        "load_signature_counts": {},
        "min_voltage_values": [],
        "max_loading_values": [],
        "loss_values": [],
        "total_load_values": [],
    }

    progress = tqdm(total=n_samples, desc="Accepted diverse IEEE-33 samples", unit="sample")
    while state["valid_samples"] < n_samples and state["total_attempted"] < max_attempts:
        remaining = n_samples - state["valid_samples"]
        current_batch_size = min(batch_size, max(remaining * 2, 1))
        plans = []
        snapshot = _history_snapshot(state)
        for _ in range(current_batch_size):
            switch_mode, load_mode, stress_case = _candidate_plan(state, rng)
            plans.append((next_seed, switch_mode, load_mode, stress_case, snapshot))
            next_seed += 1

        results = Parallel(n_jobs=n_jobs, backend="loky")(
            delayed(generate_sample)(seed, switch_mode, load_mode, stress_case, snapshot)
            for seed, switch_mode, load_mode, stress_case, snapshot in plans
        )

        accepted_records = []
        for result in results:
            state["total_attempted"] += 1
            if result.get("success", False):
                state["converged_candidates"] += 1
                accept, reason = _accept_record(result, state)
                if accept and state["valid_samples"] < n_samples:
                    accepted_records.append(result)
                    _record_acceptance(result, state)
                else:
                    state["rejected_samples"] += 1
                    state["rejection_reasons"][reason] = int(state["rejection_reasons"].get(reason, 0)) + 1
            else:
                state["rejected_samples"] += 1
                reason = result.get("reason", "unknown")
                state["rejection_reasons"][reason] = int(state["rejection_reasons"].get(reason, 0)) + 1

        if accepted_records:
            next_sample_id = write_records_to_hdf5(h5_path, accepted_records, next_sample_id)
            progress.update(len(accepted_records))
        update_hdf5_report(h5_path, state)
    progress.close()

    if state["valid_samples"] < n_samples:
        raise RuntimeError(
            f"Generated {state['valid_samples']} valid samples out of requested {n_samples} "
            f"after {state['total_attempted']} attempts. Rejections: {state['rejection_reasons']}"
        )

    report = update_hdf5_report(h5_path, state)
    print(json.dumps(report, indent=2))
    return {"h5_path": str(h5_path), **report, "rejection_reasons": state["rejection_reasons"]}

In [ ]:
# Run the production default dataset generation using the upgraded schema-v2 generator.
generation_report = generate_dataset(N_SAMPLES, H5_PATH, BATCH_SIZE, N_JOBS, overwrite=True)

## 7. Streaming Dataset Analysis with Plotly

The analysis reader scans HDF5 sample groups and collects scalar labels, global features, switch keys, and load scaling factors. It does not load node or edge feature matrices into memory.

In [ ]:
def _read_metadata_dataset(dataset):
    raw = dataset[()]
    if isinstance(raw, bytes):
        raw = raw.decode("utf-8")
    return json.loads(raw)


def collect_dataset_analysis(h5_path=H5_PATH):
    """Collect scalar summaries for interactive analysis without loading graph arrays."""
    rows = []
    load_scales = []
    attrs = {}

    with h5py.File(h5_path, "r") as h5:
        attrs = {key: h5.attrs[key] for key in h5.attrs.keys()}
        sample_names = sorted(name for name in h5.keys() if name.startswith("sample_"))
        for sample_name in tqdm(sample_names, desc="Reading scalar summaries", unit="sample"):
            group = h5[sample_name]
            labels = group["labels"][:]
            global_features = group["global_features"][:]
            metadata = _read_metadata_dataset(group["metadata"])
            load_scales.extend(metadata.get("load_scaling_factors", []))
            rows.append({
                "sample": sample_name,
                "seed": int(metadata["seed"]),
                "total_load_mw": float(global_features[0]),
                "total_loss_kw": float(labels[0]),
                "min_voltage_pu": float(labels[1]),
                "max_loading_percent": float(labels[2]),
                "violations": int(labels[3]),
                "voltage_violation_count": int(metadata.get("voltage_violation_count", 0)),
                "overload_count": int(metadata.get("overload_count", 0)),
                "switch_config_key": metadata.get("switch_config_key", ""),
            })

    analysis_df = pd.DataFrame(rows)
    load_scale_series = pd.Series(load_scales, name="load_scaling_factor", dtype=float)
    return analysis_df, load_scale_series, attrs


def print_feasibility_report(attrs):
    total = int(attrs.get("total_attempted", 0))
    valid = int(attrs.get("valid_samples", 0))
    rejected = int(attrs.get("rejected_samples", 0))
    success_rate = 100.0 * valid / max(total, 1)
    reasons = json.loads(attrs.get("rejection_reasons", "{}"))
    print("Feasibility Report")
    print("-" * 22)
    print(f"Total samples attempted : {total:,}")
    print(f"Valid samples           : {valid:,}")
    print(f"Rejected samples        : {rejected:,}")
    print(f"Success rate            : {success_rate:.2f}%")
    print(f"Rejection reasons       : {json.dumps(reasons, indent=2)}")


def plot_loss_distribution(df):
    fig = px.histogram(df, x="total_loss_kw", nbins=50, title="Total Loss Distribution", labels={"total_loss_kw": "Total loss (kW)"})
    fig.update_layout(bargap=0.02)
    fig.show()


def plot_voltage_distribution(df):
    hist = px.histogram(df, x="min_voltage_pu", nbins=50, title="Minimum Voltage Distribution", labels={"min_voltage_pu": "Minimum voltage (pu)"})
    hist.update_layout(bargap=0.02)
    hist.show()

    box = px.box(df, y="min_voltage_pu", title="Minimum Voltage Box Plot", labels={"min_voltage_pu": "Minimum voltage (pu)"})
    box.show()


def plot_loading_distribution(df):
    fig = px.histogram(df, x="max_loading_percent", nbins=50, title="Maximum Line Loading Distribution", labels={"max_loading_percent": "Maximum loading (%)"})
    fig.update_layout(bargap=0.02)
    fig.show()


def plot_switching_configurations(df):
    counts = df["switch_config_key"].value_counts()
    print(f"Number of unique switching configurations: {len(counts):,}")
    print("Top 10 most frequent configurations:")
    display(counts.head(10).rename("count").to_frame())

    top10 = counts.head(10).reset_index()
    top10.columns = ["switch_config_key", "count"]
    fig = px.bar(top10, x="switch_config_key", y="count", title="Top 10 Switching Configurations", labels={"switch_config_key": "Switch configuration", "count": "Frequency"})
    fig.update_layout(xaxis_tickangle=-45)
    fig.show()


def plot_load_scaling_distribution(load_scale_series):
    fig = px.histogram(load_scale_series.to_frame(), x="load_scaling_factor", nbins=50, title="Load Scaling Factor Distribution", labels={"load_scaling_factor": "Load perturbation factor"})
    fig.update_layout(bargap=0.02)
    fig.show()


def plot_correlation_heatmap(df):
    columns = ["total_load_mw", "total_loss_kw", "min_voltage_pu", "max_loading_percent"]
    corr = df[columns].corr()
    fig = go.Figure(data=go.Heatmap(
        z=corr.values,
        x=corr.columns,
        y=corr.index,
        colorscale="RdBu",
        zmin=-1,
        zmax=1,
        colorbar={"title": "Correlation"},
    ))
    fig.update_layout(title="Correlation Heatmap: Load, Loss, Voltage, Loading")
    fig.show()


def plot_constraint_violation_statistics(df):
    fig = px.histogram(df, x="violations", nbins=max(10, int(df["violations"].max()) + 1 if len(df) else 10), title="Constraint Violations per Sample", labels={"violations": "Voltage + overload violations"})
    fig.update_layout(bargap=0.02)
    fig.show()

## 7A. Publication-Quality Monitoring and Plotly Dashboard Upgrade

The reader below streams scalar summaries from the nested HDF5 schema and computes diversity, entropy, stress, convergence, and rejection metrics without loading graph matrices into memory.

In [ ]:
def _read_text_dataset(dataset):
    value = dataset[()]
    if isinstance(value, bytes):
        return value.decode("utf-8")
    return str(value)


def collect_dataset_analysis(h5_path=H5_PATH):
    """Stream scalar summaries from schema v2.0 without loading graph tensors."""
    rows = []
    load_scales = []
    attrs = {}
    with h5py.File(h5_path, "r") as h5:
        attrs = {key: h5.attrs[key] for key in h5.attrs.keys()}
        sample_names = sorted(name for name in h5.keys() if name.startswith("sample_"))
        seen_configs = set()
        for idx, sample_name in enumerate(tqdm(sample_names, desc="Reading quality summaries", unit="sample"), start=1):
            sample = h5[sample_name]
            metadata = json.loads(_read_text_dataset(sample["metadata"]["json"]))
            load_mode = _read_text_dataset(sample["scenario"]["load_mode"])
            config_key = _read_text_dataset(sample["topology"]["switch_config_key"])
            topology_id = _read_text_dataset(sample["topology"]["topology_id"])
            config_hash = _read_text_dataset(sample["topology"]["config_hash"])
            load_signature_value = _read_text_dataset(sample["scenario"]["load_signature"])
            load_scaling = sample["scenario"]["load_scaling_vector"][:]
            load_scales.extend(load_scaling.tolist())
            seen_configs.add(config_key)
            rows.append({
                "sample": sample_name,
                "sample_index": idx,
                "seed": int(sample["metadata"]["seed"][()]),
                "loss": float(sample["labels"]["loss"][()]),
                "min_voltage": float(sample["labels"]["min_voltage"][()]),
                "max_loading": float(sample["labels"]["max_loading"][()]),
                "total_load_mw": float(sample.attrs["total_load_mw"]),
                "validity_score": float(sample["metadata"]["validity_score"][()]),
                "switch_mode": _read_text_dataset(sample["metadata"]["switch_mode"]),
                "switch_config_key": config_key,
                "switch_config_cluster": str(abs(hash(config_key)) % 12),
                "topology_id": topology_id,
                "config_hash": config_hash,
                "load_signature": load_signature_value,
                "load_mode": load_mode,
                "stress_case": bool(sample["scenario"]["stress_case"][()]),
                "stress_label": "stress" if bool(sample["scenario"]["stress_case"][()]) else load_mode,
                "violations": int(sample.attrs["violations"]),
                "unique_configs_so_far": len(seen_configs),
                "switch_diversity_so_far": len(seen_configs) / idx,
                "voltage_violation_count": int(metadata.get("voltage_violation_count", 0)),
                "overload_count": int(metadata.get("overload_count", 0)),
            })
    return pd.DataFrame(rows), pd.Series(load_scales, name="load_scaling_factor", dtype=float), attrs


def track_distribution_metrics(dataset):
    """Compute online-style quality metrics from a scalar analysis DataFrame."""
    if isinstance(dataset, pd.DataFrame):
        df = dataset
        total = len(df)
        config_counts = df["switch_config_key"].value_counts().to_dict() if total else {}
        topology_counts = df["topology_id"].value_counts().to_dict() if total else {}
        metrics = {
            "total_samples": int(total),
            "switch_diversity_score": float(len(config_counts) / max(total, 1)),
            "topology_entropy": _entropy_from_counts(topology_counts),
            "voltage_distribution_entropy": _histogram_entropy(df["min_voltage"].to_numpy() if total else []),
            "loading_distribution_entropy": _histogram_entropy(df["max_loading"].to_numpy() if total else []),
            "stress_case_percentage": float(100.0 * df["stress_case"].mean()) if total else 0.0,
            "voltage_stats": df["min_voltage"].describe().to_dict() if total else {},
            "loading_stats": df["max_loading"].describe().to_dict() if total else {},
        }
        return metrics
    raise TypeError("track_distribution_metrics expects a pandas DataFrame produced by collect_dataset_analysis.")


def print_feasibility_report(attrs):
    total = int(attrs.get("total_attempted", 0))
    valid = int(attrs.get("valid_samples", 0))
    rejected = int(attrs.get("rejected_samples", 0))
    rejection_rate = 100.0 * rejected / max(total, 1)
    success_rate = 100.0 * valid / max(total, 1)
    reasons = json.loads(attrs.get("rejection_reasons", "{}"))
    quality_report = json.loads(attrs.get("quality_report", "{}"))
    print("Feasibility and Data Quality Report")
    print("-" * 42)
    print(f"Total candidates attempted : {total:,}")
    print(f"Valid samples generated    : {valid:,}")
    print(f"Rejected candidates        : {rejected:,}")
    print(f"Success rate               : {success_rate:.2f}%")
    print(f"Rejection rate             : {rejection_rate:.2f}%")
    print(f"Switch diversity score     : {quality_report.get('switch_diversity_score', 0.0):.3f}")
    print(f"Topology entropy           : {quality_report.get('topology_entropy', 0.0):.3f}")
    print(f"Stress case percentage     : {quality_report.get('stress_case_percentage', 0.0):.2f}%")
    print(f"Rejection reasons          : {json.dumps(reasons, indent=2)}")


def plot_loss_distribution(df):
    fig = px.histogram(df, x="loss", color="load_mode", nbins=60, title="Loss Distribution by Load Regime", labels={"loss": "Total loss (kW)"})
    fig.update_layout(bargap=0.02)
    fig.show()


def plot_voltage_distribution(df):
    hist = px.histogram(
        df,
        x="min_voltage",
        color="stress_label",
        nbins=60,
        title="Voltage Stress Distribution Colored by Load Mode",
        labels={"min_voltage": "Minimum voltage (pu)", "stress_label": "Scenario type"},
    )
    hist.add_vline(x=STRESS_VOLTAGE_THRESHOLD, line_dash="dash", line_color="red", annotation_text="0.85 pu stress threshold")
    hist.show()
    box = px.box(df, x="load_mode", y="min_voltage", color="stress_case", title="Minimum Voltage by Load Regime", labels={"min_voltage": "Minimum voltage (pu)"})
    box.show()


def plot_loading_distribution(df):
    fig = px.histogram(df, x="max_loading", color="stress_label", nbins=60, title="Maximum Line Loading Distribution", labels={"max_loading": "Maximum loading (%)"})
    fig.add_vline(x=STRESS_LOADING_THRESHOLD, line_dash="dash", line_color="red", annotation_text="90% stress threshold")
    fig.show()


def plot_switching_configurations(df):
    counts = df["switch_config_key"].value_counts()
    print(f"Number of unique switching configurations: {len(counts):,}")
    print(f"Switch diversity score: {len(counts) / max(len(df), 1):.3f}")
    print("Top 10 most frequent configurations:")
    print(counts.head(10).rename("count").to_string())

    pareto = counts.reset_index()
    pareto.columns = ["switch_config_key", "count"]
    pareto["rank"] = np.arange(1, len(pareto) + 1)
    pareto["cumulative_percent"] = 100.0 * pareto["count"].cumsum() / max(pareto["count"].sum(), 1)
    fig = go.Figure()
    fig.add_bar(x=pareto["rank"], y=pareto["count"], name="Frequency")
    fig.add_trace(go.Scatter(x=pareto["rank"], y=pareto["cumulative_percent"], yaxis="y2", mode="lines+markers", name="Cumulative %"))
    fig.update_layout(
        title="Switch Configuration Frequency Pareto Plot",
        xaxis_title="Configuration rank",
        yaxis_title="Frequency",
        yaxis2={"title": "Cumulative %", "overlaying": "y", "side": "right", "range": [0, 100]},
        bargap=0.02,
    )
    fig.show()


def plot_load_scaling_distribution(load_scale_series):
    fig = px.histogram(load_scale_series.to_frame(), x="load_scaling_factor", nbins=70, title="Balanced Load Scaling Distribution", labels={"load_scaling_factor": "Load perturbation factor"})
    fig.update_layout(bargap=0.02)
    fig.show()


def plot_correlation_heatmap(df):
    columns = ["total_load_mw", "loss", "min_voltage", "max_loading", "validity_score"]
    corr = df[columns].corr()
    fig = go.Figure(data=go.Heatmap(z=corr.values, x=corr.columns, y=corr.index, colorscale="RdBu", zmin=-1, zmax=1, colorbar={"title": "Correlation"}))
    fig.update_layout(title="Correlation Heatmap: Load, Loss, Voltage, Loading, Validity")
    fig.show()


def plot_constraint_violation_statistics(df):
    fig = px.histogram(df, x="violations", color="stress_case", nbins=30, title="Constraint Violations per Sample", labels={"violations": "Voltage + overload violations"})
    fig.update_layout(bargap=0.02)
    fig.show()


def plot_topology_diversity_trend(df):
    fig = px.line(df, x="sample_index", y="switch_diversity_so_far", title="Topology Diversity Trend During Dataset Generation", labels={"sample_index": "Accepted sample index", "switch_diversity_so_far": "Unique configs / samples"})
    fig.add_hline(y=MIN_SWITCH_DIVERSITY, line_dash="dash", line_color="red", annotation_text="minimum 0.30")
    fig.add_hline(y=TARGET_SWITCH_DIVERSITY, line_dash="dash", line_color="green", annotation_text="target 0.50")
    fig.show()


def plot_loss_vs_load_scatter(df):
    fig = px.scatter(
        df,
        x="total_load_mw",
        y="loss",
        color="switch_config_cluster",
        symbol="load_mode",
        hover_data=["sample", "min_voltage", "max_loading", "validity_score", "topology_id"],
        title="Loss vs Load Colored by Switching Configuration Cluster",
        labels={"total_load_mw": "Total load (MW)", "loss": "Loss (kW)"},
    )
    fig.show()


def plot_feasibility_boundary(df):
    fig = px.scatter(
        df,
        x="min_voltage",
        y="max_loading",
        color="validity_score",
        symbol="stress_case",
        hover_data=["sample", "load_mode", "loss", "switch_mode"],
        title="Feasibility Boundary: Minimum Voltage vs Maximum Loading",
        labels={"min_voltage": "Minimum voltage (pu)", "max_loading": "Maximum loading (%)"},
        color_continuous_scale="Viridis",
    )
    fig.add_vline(x=0.95, line_dash="dot", line_color="orange", annotation_text="0.95 pu nominal limit")
    fig.add_vline(x=STRESS_VOLTAGE_THRESHOLD, line_dash="dash", line_color="red", annotation_text="0.85 pu stress")
    fig.add_hline(y=STRESS_LOADING_THRESHOLD, line_dash="dash", line_color="red", annotation_text="90% loading")
    fig.add_hline(y=100.0, line_dash="dot", line_color="orange", annotation_text="100% overload")
    fig.show()


def print_final_data_quality_report(df, attrs):
    metrics = track_distribution_metrics(df)
    total_attempted = int(attrs.get("total_attempted", 0))
    valid = int(attrs.get("valid_samples", len(df)))
    rejected = int(attrs.get("rejected_samples", 0))
    print("Final Dataset Quality Report")
    print("=" * 36)
    print(f"Total samples generated       : {total_attempted:,}")
    print(f"Valid samples                 : {valid:,}")
    print(f"Rejection rate                : {100.0 * rejected / max(total_attempted, 1):.2f}%")
    print(f"Switch diversity score        : {metrics['switch_diversity_score']:.3f}")
    print(f"Topology entropy              : {metrics['topology_entropy']:.3f}")
    print(f"Voltage distribution entropy  : {metrics['voltage_distribution_entropy']:.3f}")
    print(f"Loading distribution entropy  : {metrics['loading_distribution_entropy']:.3f}")
    print(f"Stress case percentage        : {metrics['stress_case_percentage']:.2f}%")
    print("Voltage stats:")
    print(pd.Series(metrics["voltage_stats"]).to_string())
    print("Loading stats:")
    print(pd.Series(metrics["loading_stats"]).to_string())

In [ ]:
analysis_df, load_scale_series, dataset_attrs = collect_dataset_analysis(H5_PATH)

print_feasibility_report(dataset_attrs)

if len(analysis_df) == 0:
    raise RuntimeError("No valid samples were generated; analysis cannot continue.")

plot_loss_distribution(analysis_df)
plot_voltage_distribution(analysis_df)
plot_loading_distribution(analysis_df)
plot_switching_configurations(analysis_df)
plot_topology_diversity_trend(analysis_df)
plot_load_scaling_distribution(load_scale_series)
plot_loss_vs_load_scatter(analysis_df)
plot_feasibility_boundary(analysis_df)
plot_correlation_heatmap(analysis_df)
plot_constraint_violation_statistics(analysis_df)
print_final_data_quality_report(analysis_df, dataset_attrs)

analysis_df.describe(include="all")

## 8. Dataset Summary for Graph Transformer Training

In [ ]:
def print_dataset_summary(h5_path=H5_PATH):
    with h5py.File(h5_path, "r") as h5:
        sample_names = sorted(name for name in h5.keys() if name.startswith("sample_"))
        if len(sample_names) == 0:
            raise RuntimeError("HDF5 file contains no sample groups.")

        first = h5[sample_names[0]]
        node_shape = first["graph"]["node_features"].shape
        edge_shape = first["graph"]["edge_features"].shape
        global_shape = first["graph"]["global_features"].shape
        label_names = json.loads(h5.attrs["label_names"])
        quality_report = json.loads(h5.attrs.get("quality_report", "{}"))

        print("Dataset Ready for Graph Transformer Training")
        print("=" * 56)
        print(f"HDF5 path          : {h5_path}")
        print(f"Schema version     : {h5.attrs.get('schema_version', 'unknown')}")
        print(f"Number of samples  : {len(sample_names):,}")
        print(f"Node features      : shape={node_shape}, names={json.loads(h5.attrs['node_feature_names'])}")
        print(f"Edge features      : shape={edge_shape}, names={json.loads(h5.attrs['edge_feature_names'])}")
        print(f"Global features    : shape={global_shape}, names={json.loads(h5.attrs['global_feature_names'])}")
        print(f"Labels             : names={label_names}")
        print(f"Switch diversity   : {quality_report.get('switch_diversity_score', 0.0):.3f}")
        print(f"Topology entropy   : {quality_report.get('topology_entropy', 0.0):.3f}")
        print(f"Stress percentage  : {quality_report.get('stress_case_percentage', 0.0):.2f}%")
        print("\nLabel definitions")
        print("- loss: total active branch loss in kW")
        print("- min_voltage: minimum bus voltage magnitude in per-unit")
        print("- max_loading: maximum line loading percentage")
        print("\nHDF5 storage structure")
        print("/sample_000001/")
        print("    graph/")
        print("        node_features      float32 [num_buses, 7]")
        print("        edge_features      float32 [num_lines, 5]")
        print("        global_features    float32 [5]")
        print("    topology/")
        print("        config_hash        string")
        print("        topology_id        string")
        print("        switch_config_key  string")
        print("    scenario/")
        print("        load_mode          string")
        print("        load_scaling_vector float32 [num_buses]")
        print("        load_signature     string")
        print("        stress_case        bool")
        print("    labels/")
        print("        loss               float32")
        print("        min_voltage        float32")
        print("        max_loading        float32")
        print("    metadata/")
        print("        seed               int64")
        print("        validity_score     float32")
        print("        switch_mode        string")
        print("        json               string")


print_dataset_summary(H5_PATH)